In [49]:
import os
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torchvision
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader, Subset

In [50]:
import os
import pandas as pd

df = pd.read_csv("labels.csv")

image_files = set(os.listdir("images"))

missing = []

for f in df["filename"]:
    if f not in image_files:
        missing.append(f)

print("Missing images:", len(missing))
print(missing[:10])

Missing images: 0
[]


In [51]:
df = df[df["filename"].isin(image_files)]
df = df.reset_index(drop=True)

df.to_csv("clean_labels.csv", index=False)

print("Clean dataset size:", len(df))

Clean dataset size: 1999


In [52]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA GeForce RTX 3050 6GB Laptop GPU


In [53]:
class CarDamageDataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None):
        self.data = pd.read_csv(csv_file)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_name = self.data.iloc[idx]["filename"]
        label = self.data.iloc[idx]["label"]

        img_path = os.path.join(self.img_dir, img_name)
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label

In [54]:
transform = transforms.Compose([
    transforms.Resize((128,128)),   # FAST
    transforms.ToTensor()
])

In [55]:
dataset = CarDamageDataset(
    csv_file="clean_labels.csv",
    img_dir="images/",
    transform=transform
)

subset_size = min(2000, len(dataset))
dataset = Subset(dataset, range(subset_size))

train_loader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0   # 🔥 FIX
)

print("Dataset loaded:", len(dataset))

Dataset loaded: 1999


In [56]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
model = torchvision.models.mobilenet_v2(pretrained=True)

# Freeze layers
for param in model.parameters():
    param.requires_grad = False

# Replace classifier
model.classifier[1] = nn.Linear(model.last_channel, 2)

model = model.to(device)

Using device: cuda


C:\Users\acer\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\acer\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [57]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.classifier.parameters(), lr=0.001)

In [58]:
epochs = 3   # FAST

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

Epoch 1, Loss: 15.5047
Epoch 2, Loss: 9.4690
Epoch 3, Loss: 9.3086


In [59]:
torch.save(model.state_dict(), "image_fraud_model_fast.pth")
print("Model saved!")

Model saved!


In [62]:
model.eval()

classes = ["not_fraud", "fraud"]

def predict_image(img_path):
    img = Image.open(img_path).convert("RGB")
    img = transform(img).unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(img)
        pred = torch.argmax(output, 1).item()

    return classes[pred]

# Test
img_path = "images/test.jpg"
print(predict_image(img_path))

FileNotFoundError: [Errno 2] No such file or directory: 'images/test.jpg'